# 13 · Grouped Taxonomy Experiment

**Purpose.** Retrain the best architecture with Contact Dermatitis and Eczema merged into one 'Eczema / Dermatitis' class, and compare with the 6-class model.

| | |
|---|---|
| **Reads** | `data/splits/*`, `best_models.json` |
| **Writes** | `models/grouped/*`, `reports/results/grouped_taxonomy.json` |
| **Run time (Colab T4)** | 30-60 min |
| **Next notebook** | 14 · Export |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dermalens.data import make_loader, resolve_images
from dermalens.utils import get_device, load_json, save_json

info = load_json(paths.splits / "class_info.json")       # classes + weights frozen in notebook 05
CLASSES = info["classes"]                                  # 6 targets + "Other"
WEIGHTS = [info["class_weights"][name] for name in CLASSES]
train = pd.read_csv(paths.splits / "train.csv")
val = pd.read_csv(paths.splits / "val.csv")
IMAGE_ROOT, PATH_COL = resolve_images(train, paths)        # cache if present, else raw files
DEVICE = get_device()
IMAGE_SIZE = CONFIG["data"]["image_size"]
BATCH, WORKERS = CONFIG["train"]["batch_size"], CONFIG["train"]["num_workers"]
USE_META, COLOR = CONFIG["data"]["use_metadata"], CONFIG["data"]["color_constancy"]


def loader(rows, train_mode=False, size=IMAGE_SIZE, use_meta=USE_META, classes=None, label_col="label",
           seed=0):
    """One DataLoader with the project settings (augmentation + metadata dropout only when training)."""
    return make_loader(rows, IMAGE_ROOT, classes or CLASSES, size, train=train_mode, batch_size=BATCH,
                       num_workers=WORKERS, path_col=PATH_COL, label_col=label_col, use_metadata=use_meta,
                       meta_dropout=CONFIG["data"]["meta_dropout"], color_constancy=COLOR, seed=seed)


print(f"train {len(train)} | val {len(val)} | classes {len(CLASSES)} | device {DEVICE} | "
      f"images from {IMAGE_ROOT} ({PATH_COL})")

In [ ]:
import time
from dermalens.evaluate import headline_metrics, predict_logits
from dermalens.model import build_model, count_parameters, zoo_entry
from dermalens.train import fit, model_from_checkpoint, train_config_from
from dermalens.utils import set_seed

TRAIN_CFG = train_config_from(CONFIG["train"])


def train_run(model_name, seed, use_meta, folder, rows=None, classes=None, weights=None, label_col="label"):
    """Train one model (or reuse it if done.json exists). Returns its validation record."""
    run_dir = paths.models / folder / f"{model_name}_{'meta' if use_meta else 'img'}_seed{seed}"
    if (run_dir / "done.json").exists():
        print("skip (done):", run_dir.name)
        return load_json(run_dir / "done.json")
    rows = rows if rows is not None else (train, val)
    classes, weights = classes or CLASSES, weights or WEIGHTS
    set_seed(seed)
    size = IMAGE_SIZE if zoo_entry(model_name)[2] == "cnn" else min(IMAGE_SIZE, 224)
    model = build_model(model_name, len(classes), CONFIG["model"]["pretrained"], use_meta,
                        CONFIG["model"]["drop_rate"], image_size=size)
    facts = {"model_name": model_name, "classes": classes, "image_size": size, "seed": seed,
             "use_metadata": use_meta}
    started = time.time()
    history = fit(model, loader(rows[0], True, size, use_meta, classes, label_col, seed),
                  loader(rows[1], False, size, use_meta, classes, label_col), weights, TRAIN_CFG, DEVICE,
                  run_dir, facts, seed=seed)
    history.to_csv(run_dir / "history.csv", index=False)
    best, _ = model_from_checkpoint(run_dir / "best.pt", DEVICE)
    logits, y_true = predict_logits(best, loader(rows[1], False, size, use_meta, classes, label_col), DEVICE)
    record = {"model": model_name, "seed": seed, "use_metadata": use_meta, "image_size": size,
              "checkpoint": str((run_dir / "best.pt").relative_to(paths.root)),
              "params_m": round(count_parameters(best)[0] / 1e6, 1),
              "minutes": round((time.time() - started) / 60, 1),
              **{k: round(float(v), 4) for k, v in headline_metrics(y_true, logits.argmax(1)).items()}}
    save_json(record, run_dir / "done.json")
    print(record)
    return record

from dermalens.labels import group_label
from dermalens.splits import compute_class_weights

test = pd.read_csv(paths.splits / "test.csv")
regroup = lambda rows: rows.assign(label=rows["label"].map(group_label))
g_train, g_val, g_test = regroup(train), regroup(val), regroup(test)
G_CLASSES = sorted(set(g_train["label"]) - {"Other"}) + (["Other"] if "Other" in CLASSES else [])
weights = compute_class_weights(g_train, G_CLASSES)
print(G_CLASSES)

**Why:** in a clinical photo, allergic/irritant contact dermatitis and atopic eczema share redness,
scaling and lichenification; the distinction often needs patch testing or history. Many published
systems group them. If the grouped model is much better, the report should present both results and
explain the trade-off (a coarser but more reliable answer).

In [ ]:
best = load_json(paths.results / "best_models.json")["members"][0]
record = train_run(best["model"], CONFIG["seed"], USE_META, "grouped", rows=(g_train, g_val),
                   classes=G_CLASSES, weights=[weights[c] for c in G_CLASSES])
model, facts = model_from_checkpoint(paths.root / record["checkpoint"], DEVICE)
logits, y_true = predict_logits(model, loader(g_test, size=facts["image_size"], classes=G_CLASSES), DEVICE,
                                tta=True)
known = np.array([G_CLASSES[i] != "Other" for i in y_true])
known_idx = [i for i, c in enumerate(G_CLASSES) if c != "Other"]
pred = np.array(known_idx)[logits[known][:, known_idx].argmax(1)]
grouped_test = {k: round(float(v), 4) for k, v in headline_metrics(y_true[known], pred).items()}
six_class = load_json(paths.results / "test_results.json")["headline_known_classes"]
comparison = {"grouped_5_class_test": grouped_test,
              "six_class_test": {k: v["value"] for k, v in six_class.items()}}
save_json(comparison, paths.results / "grouped_taxonomy.json")
display(pd.DataFrame(comparison))